# E-Commerce Analytics 360
## Étape 4 — Power BI Design Guide

> **Prérequis** : avoir complété les étapes 1, 2 et 3. Les fichiers CSV exportés depuis Colab doivent être disponibles sur ton PC.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Power BI Desktop |
| **Durée estimée** | 3h à 4h |

> 💡 **Ce notebook est un guide de conception.** Il ne contient pas de code Python. Tu vas travailler directement dans Power BI Desktop en suivant les instructions ci-dessous.

---
## 1. Sources de données à importer

Les données ont été préparées dans les étapes précédentes et exportées en CSV depuis Colab. Tu as besoin des fichiers suivants sur ton PC :

| Fichier | Produit dans |
|---|---|
| `fact_ecommerce_analytics.csv` | Étape 2 |
| `clients_rfm_segments.csv` | Étape 3 |

### Comment les récupérer depuis Colab
Dans Colab → panneau latéral gauche → icône **dossier** → clic droit sur le fichier → **Download**.

### Importer dans Power BI
1. Ouvre Power BI Desktop
2. **Obtenir les données** → **Texte/CSV**
3. Sélectionne chaque fichier
4. Choisis le mode **Import**
5. Vérifie les types de colonnes dans Power Query avant de charger

---
## 2. Modèle de données recommandé

Nous utilisons un **modèle en étoile** à partir des deux fichiers CSV.

### Tables

| Table | Source | Rôle |
|---|---|---|
| `fact_ecommerce_analytics` | CSV étape 2 | Table de faits principale |
| `clients_rfm_segments` | CSV étape 3 | Dimension client enrichie |
| `dim_date` | Créée dans Power BI | Dimension calendrier |

### Créer la dimension date dans Power BI
Dans Power Query → **Nouvelle requête vide** → colle ce code M :

```
let
    StartDate = #date(2023, 1, 1),
    EndDate   = Date.From(DateTime.LocalNow()),
    NbDays    = Duration.Days(EndDate - StartDate) + 1,
    Dates     = List.Dates(StartDate, NbDays, #duration(1,0,0,0)),
    Table     = Table.FromList(Dates, Splitter.SplitByNothing(), {"Date"}),
    TypedDate = Table.TransformColumnTypes(Table, {{"Date", type date}}),
    Year      = Table.AddColumn(TypedDate, "Année",     each Date.Year([Date]),   Int64.Type),
    Month     = Table.AddColumn(Year,      "Mois",      each Date.Month([Date]),  Int64.Type),
    MonthName = Table.AddColumn(Month,     "Mois Nom",  each Date.MonthName([Date])),
    Quarter   = Table.AddColumn(MonthName, "Trimestre", each "T" & Text.From(Date.QuarterOfYear([Date])))
in
    Quarter
```

### Relations à créer

- `fact_ecommerce_analytics[customer_id]` → `clients_rfm_segments[customer_id]`
- `fact_ecommerce_analytics[order_date]` → `dim_date[Date]`

### ⚠️ Conseils
- Relations en **One-to-Many** uniquement
- Sens de filtre simple (évite le bidirectionnel)
- Marque `dim_date` comme **table de dates** dans les propriétés

---
## 3. Structure globale du dashboard

Le dashboard est composé de **6 pages** :

| Page | Question business principale |
|---|---|
| 1. Overview | Comment évolue la performance globale ? |
| 2. Produits | Quels produits génèrent le plus de valeur ? |
| 3. Clients | Quels segments contribuent le plus ? |
| 4. Funnel digital | Où perd-on les prospects ? |
| 5. Satisfaction client | Quel est le niveau de satisfaction ? |
| 6. Insights & recommandations | Quelles actions prioritaires lancer ? |

### Principes de design
- Une page = une question business principale
- KPIs toujours en haut
- Filtres à gauche ou en haut
- Cohérence de couleurs et de polices sur toutes les pages

---
## 4. Page 1 — Overview

**Question business** : Comment évolue la performance globale de l'entreprise ?

### KPI Cards (ligne 1)
CA total · Marge totale · Nombre de commandes · Nombre de clients · Panier moyen · Note moyenne

### Visuels (lignes 2 et 3)
- Courbe : évolution mensuelle du CA
- Histogramme : CA par catégorie
- Bar chart : CA par canal
- Donut : répartition par moyen de paiement

### Colonnes utiles
`order_date`, `revenue`, `margin`, `order_id`, `customer_id`, `canal`, `payment_method`, `rating`, `categorie`

---
## 5. Page 2 — Produits

**Question business** : Quels produits et catégories génèrent le plus de valeur ?

### KPI Cards
Quantité totale vendue · Revenu total · Marge totale

### Visuels
- Bar chart : Top 10 produits par CA
- Bar chart : Top catégories par CA
- Scatter plot : revenu vs marge par produit
- Table : produit, catégorie, quantité, revenu, marge (avec mise en forme conditionnelle pour les produits peu rentables)

### Colonnes utiles
`nom_produit`, `categorie`, `marque`, `quantity`, `revenue`, `margin`, `cost`

---
## 6. Page 3 — Clients

**Question business** : Quels segments et zones géographiques contribuent le plus ?

### KPI Cards
Clients actifs · Revenu moyen/client · Commandes moyennes/client

### Visuels
- Bar chart : CA par segment client
- Bar chart ou Map : CA par pays / ville
- Table : Top 10 clients
- Bar chart : répartition des segments RFM (`segment_rfm` depuis `clients_rfm_segments`)
- Scatter : Recency vs Monetary par segment RFM

### Colonnes utiles
`customer_id`, `segment_client`, `pays`, `ville`, `revenue`, `segment_rfm`, `recency`, `frequency`, `monetary`

> 💡 C'est ici qu'on exploite le fichier `clients_rfm_segments.csv` de l'étape 3.

---
## 7. Page 4 — Funnel digital

**Question business** : Comment les utilisateurs progressent-ils dans le tunnel d'achat ?

### KPI Cards
Nb vues · Nb ajouts panier · Nb achats · Taux de conversion global

### Visuels
- Funnel chart : view → add_to_cart → purchase
- Bar chart : conversion par canal
- Bar chart : conversion par device
- Table : canal, device, vues, panier, achats, taux de conversion

### Colonnes utiles
`canal`, `device`, colonnes de funnel calculées en DAX (voir section mesures)

---
## 8. Page 5 — Satisfaction client

**Question business** : Quel est le niveau de satisfaction et quels produits posent problème ?

### KPI Cards
Note moyenne globale · Nombre total d'avis · Nb produits avec note < 3

### Visuels
- Bar chart : produits avec les plus faibles notes
- Histogramme : distribution des ratings (1 à 5)
- Courbe : évolution mensuelle de la note moyenne
- Table : produit, catégorie, note moyenne, nombre d'avis

### Colonnes utiles
`rating`, `nom_produit`, `categorie`, `review_date`, `order_date`

---
## 9. Page 6 — Insights & recommandations

**Question business** : Quelles actions prioritaires faut-il lancer ?

> Cette page est une **page de synthèse** destinée à un décideur. Elle est intentionnellement peu graphique.

### Structure recommandée

**Bloc 1 — Faits marquants**  
Exemples à compléter avec tes résultats réels :
- Le CA est concentré sur ___ catégories représentant ___% du total
- Certains produits vendent beaucoup mais ont une marge < ___% 
- Le segment ___ génère ___% du revenu malgré ___% des clients
- Le funnel montre une perte de ___% entre ajout panier et achat
- ___ produits cumulent note < 3 et volume élevé → risque image

**Bloc 2 — Recommandations**
- Optimiser les produits à forte marge
- Améliorer l'expérience checkout
- Renforcer les actions sur les canaux les plus convertissants
- Traiter les produits les plus critiqués en priorité

**Bloc 3 — Priorités**

| Horizon | Action |
|---|---|
| Court terme (< 1 mois) | *(à compléter)* |
| Moyen terme (1-3 mois) | *(à compléter)* |
| Quick wins | *(à compléter)* |

---
## 10. Mesures DAX principales

Crée ces mesures dans une table dédiée `_Mesures` dans Power BI.

### KPIs commerciaux

```dax
CA Total = SUM(fact_ecommerce_analytics[revenue])

Marge Totale = SUM(fact_ecommerce_analytics[margin])

Taux de Marge = DIVIDE([Marge Totale], [CA Total], 0)

Nb Commandes = DISTINCTCOUNT(fact_ecommerce_analytics[order_id])

Nb Clients = DISTINCTCOUNT(fact_ecommerce_analytics[customer_id])

Panier Moyen = DIVIDE([CA Total], [Nb Commandes], 0)

Quantité Vendue = SUM(fact_ecommerce_analytics[quantity])
```

### Satisfaction

```dax
Note Moyenne = AVERAGE(fact_ecommerce_analytics[rating])

Nb Avis = COUNTROWS(FILTER(fact_ecommerce_analytics, NOT(ISBLANK(fact_ecommerce_analytics[rating]))))

Produits Critiques =
CALCULATE(
    DISTINCTCOUNT(fact_ecommerce_analytics[product_id]),
    AVERAGE(fact_ecommerce_analytics[rating]) < 3
)
```

### Funnel digital

```dax
Nb Vues =
CALCULATE(
    COUNTROWS(fact_ecommerce_analytics),
    fact_ecommerce_analytics[page] = "product"
)

Taux de Conversion = DIVIDE([Nb Commandes], [Nb Vues], 0)
```

### Variation temporelle

```dax
CA Mois Précédent =
CALCULATE(
    [CA Total],
    DATEADD(dim_date[Date], -1, MONTH)
)

Variation CA % =
DIVIDE([CA Total] - [CA Mois Précédent], [CA Mois Précédent], 0)
```

---
## 11. Filtres recommandés

### Filtres globaux (toutes les pages)
Année · Mois · Pays · Segment client · Catégorie produit

### Filtres spécifiques par page

| Page | Filtres additionnels |
|---|---|
| Produits | Marque, Canal |
| Clients | Segment RFM, Ville |
| Funnel | Device, Canal |
| Satisfaction | Note (slider 1-5) |

> **Conseil UX** : utilise un panneau de filtres latéral masquable plutôt que d'empiler les slicers sur la page.

---
## 12. Bonnes pratiques de design

**✅ À faire**
- Titres clairs sur chaque visuel
- Toujours afficher les unités (€, %, nb)
- Hiérarchie visuelle simple (KPI → graphique → détail)
- Couleurs utilisées avec parcimonie et cohérence
- KPIs clés toujours visibles en haut de page

**❌ À éviter**
- Plus de 5-6 visuels sur une seule page
- Trop de couleurs différentes
- Tables avec trop de colonnes
- KPIs sans contexte (toujours afficher la variation)

---
## 13. Storytelling — ordre de lecture

Le dashboard doit raconter une histoire. L'ordre de lecture conseillé :

1. **Vue globale** — quelle est la santé de l'entreprise ?
2. **Produits** — ce qui génère la performance
3. **Clients** — qui génère la performance
4. **Funnel** — où perd-on les clients
5. **Satisfaction** — quel est le risque image
6. **Insights** — que fait-on maintenant ?

> **📌 Point pédagogique** :  
> L'apprenant ne doit pas seulement savoir créer un dashboard.  
> Il doit être capable de répondre à la question :  
> **"Que doit faire l'entreprise maintenant ?"**

---
## 14. Checklist de validation

- [ ] CSV importés sans erreur dans Power BI
- [ ] Dimension date créée et marquée correctement
- [ ] Relations définies (fact ↔ clients_rfm, fact ↔ dim_date)
- [ ] Toutes les mesures DAX créées et testées
- [ ] Page Overview complète avec 6 KPIs
- [ ] Page Produits avec scatter revenu vs marge
- [ ] Page Clients avec segmentation RFM intégrée
- [ ] Page Funnel avec visualisation du tunnel
- [ ] Page Satisfaction avec évolution mensuelle
- [ ] Page Insights rédigée avec recommandations chiffrées
- [ ] Filtres globaux fonctionnels sur toutes les pages

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.